# Notebook 08 — Stage 4 Layer 2 Phase 1: Jerby-Arnon-New cohort load + Tsoi marker preview

Stage 4 C1 Layer 2 (per `docs/stage4_dataset_schema.md` §5):
True independent validation of Stage 3 P1 dataset-level NGFR claim.
Apply Stage 2-style pipeline (Harmony) on 4199 Tirosh-overlap cells per entry h).

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import anndata as ad
from pathlib import Path
sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=80)

## Load full Jerby-Arnon dataset

In [2]:
# Load counts (genes x cells)
counts_path = Path('../data/raw/jerby_arnon/GSE115978_counts.csv.gz')
ann_path = Path('../data/raw/jerby_arnon/GSE115978_cell.annotations.csv.gz')

print('Loading counts CSV...')
counts_df = pd.read_csv(counts_path, compression='gzip', index_col=0)
print(f'Counts shape (genes x cells): {counts_df.shape}')

print('Loading annotations...')
ann_df = pd.read_csv(ann_path, compression='gzip', index_col=0)
print(f'Annotations shape: {ann_df.shape}')
print(f'Annotation columns: {ann_df.columns.tolist()}')

Loading counts CSV...


Counts shape (genes x cells): (23686, 7186)
Loading annotations...
Annotations shape: (7186, 6)
Annotation columns: ['samples', 'cell.types', 'treatment.group', 'Cohort', 'no.of.genes', 'no.of.reads']


In [3]:
# Sanity: verify counts cell IDs match annotations index exactly
counts_cells = set(counts_df.columns)
ann_cells = set(ann_df.index)
print(f'Counts cells: {len(counts_cells)}')
print(f'Ann cells: {len(ann_cells)}')
print(f'Intersection: {len(counts_cells & ann_cells)}')
print(f'In counts but not ann: {len(counts_cells - ann_cells)}')
print(f'In ann but not counts: {len(ann_cells - counts_cells)}')
assert counts_cells == ann_cells, 'Cell ID mismatch'
print('PASSED cell ID match check')

Counts cells: 7186
Ann cells: 7186
Intersection: 7186
In counts but not ann: 0
In ann but not counts: 0
PASSED cell ID match check


In [4]:
# Build full AnnData (cells x genes, scanpy convention)
adata_full = ad.AnnData(
    X=counts_df.T.values,  # transpose to cells x genes
    obs=ann_df.loc[counts_df.columns],  # ann index match counts columns
    var=pd.DataFrame(index=counts_df.index),  # gene index
)
adata_full.X = adata_full.X.astype(np.float32)
print(f'Full AnnData: {adata_full.shape}')
print(f'obs columns: {adata_full.obs.columns.tolist()}')
print(f'obs head:')
print(adata_full.obs.head())

Full AnnData: (7186, 23686)
obs columns: ['samples', 'cell.types', 'treatment.group', 'Cohort', 'no.of.genes', 'no.of.reads']
obs head:
                                                samples cell.types  \
cy78_CD45_neg_1_B04_S496_comb                     Mel78        Mal   
cy79_p4_CD45_neg_PDL1_neg_E11_S1115_comb          Mel79        Mal   
CY88_5_B10_S694_comb                              Mel88        Mal   
cy79_p1_CD45_neg_PDL1_pos_AS_C1_R1_F07_S67_comb   Mel79        Mal   
cy78_CD45_neg_3_H06_S762_comb                     Mel78        Mal   

                                                 treatment.group  Cohort  \
cy78_CD45_neg_1_B04_S496_comb                     post.treatment  Tirosh   
cy79_p4_CD45_neg_PDL1_neg_E11_S1115_comb         treatment.naive  Tirosh   
CY88_5_B10_S694_comb                              post.treatment  Tirosh   
cy79_p1_CD45_neg_PDL1_pos_AS_C1_R1_F07_S67_comb  treatment.naive  Tirosh   
cy78_CD45_neg_3_H06_S762_comb                     post.treatmen

## Layer 2 subset selection

Per schema doc §6: Layer 2 = New cohort cells, with implementation note 'For malignant subset analyses: filter by `cell.types == Mal`'.

Determine final Layer 2 cell count by counting 3 possible subsets, then make informed choice.

In [5]:
# Count 3 possible Layer 2 subsets
n_new_all = (adata_full.obs['Cohort'] == 'New').sum()
n_new_mal = ((adata_full.obs['Cohort'] == 'New') &
             (adata_full.obs['cell.types'] == 'Mal')).sum()
n_new_excl_uncertain = ((adata_full.obs['Cohort'] == 'New') &
                         (adata_full.obs['cell.types'] != '?')).sum()

print(f'Cohort == New (all cell types): {n_new_all}')
print(f'Cohort == New & cell.types == Mal: {n_new_mal}')
print(f'Cohort == New & cell.types != ?: {n_new_excl_uncertain}')

# Per cell.types breakdown in New cohort
print('\nNew cohort cell.types distribution:')
print(adata_full.obs[adata_full.obs['Cohort'] == 'New']['cell.types'].value_counts())

Cohort == New (all cell types): 2987
Cohort == New & cell.types == Mal: 860
Cohort == New & cell.types != ?: 2881

New cohort cell.types distribution:
cell.types
Mal           860
T.CD8         718
T.CD4         352
T.cell        310
Macrophage    261
B.cell        257
?             106
CAF            47
Endo.          45
NK             31
Name: count, dtype: int64


In [6]:
# Layer 2: New cohort ∩ malignant
mask = (adata_full.obs['Cohort'] == 'New') & (adata_full.obs['cell.types'] == 'Mal')
adata_layer2 = adata_full[mask].copy()
print(f'Layer 2 shape: {adata_layer2.shape}')
print(f'Sample distribution (Mel### -> # cells):')
print(adata_layer2.obs['samples'].value_counts())
print(f'Treatment group distribution:')
print(adata_layer2.obs['treatment.group'].value_counts())

# Save Layer 2 AnnData (gitignored, too large for git)
Path('../data/processed').mkdir(parents=True, exist_ok=True)
adata_layer2.write('../data/processed/jerby_arnon_layer2_raw_counts.h5ad')
print('Saved Layer 2 raw counts AnnData')

Layer 2 shape: (860, 23686)
Sample distribution (Mel### -> # cells):
samples
Mel102      169
Mel103      137
Mel110      123
Mel194       96
Mel106       94
Mel129pa     92
Mel98        76
Mel112       28
Mel128       26
Mel105       14
Mel121.1      5
Name: count, dtype: int64
Treatment group distribution:
treatment.group
post.treatment     563
treatment.naive    297
Name: count, dtype: int64


Saved Layer 2 raw counts AnnData


## Normalization

Jerby-Arnon counts are raw integer counts (Smart-seq2 read counts).
Apply standard scanpy normalization: `normalize_total(target_sum=1e6)` + `log1p`.

**Critical:** Tirosh data uses `log2(TPM/10+1)` convention; Jerby-Arnon-New (this
notebook) is normalized with `normalize_total(1e6) + log1p`. Marker mean
comparisons across the two datasets are **not** directly interpretable in
absolute terms. Use *distribution shape* and *% expressing* for cross-dataset
comparison (per Stage 4 Phase 1.2 forward fix).

In [7]:
sc.pp.normalize_total(adata_layer2, target_sum=1e6)
sc.pp.log1p(adata_layer2)
print(f'After normalize_total(1e6) + log1p, X dtype: {adata_layer2.X.dtype}')
print(f'X mean: {adata_layer2.X.mean():.3f}')
print(f'X max: {adata_layer2.X.max():.3f}')

normalizing counts per cell


    finished (0:00:00)


After normalize_total(1e6) + log1p, X dtype: float32
X mean: 0.778
X max: 12.907


## Tsoi marker distribution preview

Stage 3 mini-report P1 hypothesis: NGFR signal is uniformly weak across
malignant clusters in Tirosh, making Transitory and Neural crest-like states
unrecoverable. P1 framed this as a *dataset-level* limitation.

Layer 2 test: in Jerby-Arnon-New malignant cells, does NGFR show:
- A high subgroup (cells > some expression threshold)? → P1 may be too strong
- Uniformly weak? → P1 confirmed

Comparison metric (per Phase 1.2 forward fix):
- `pct_expressing` (cells with > 0 expression): cross-dataset comparable
- Distribution shape (unimodal weak vs bimodal): more informative than mean
- Mean reported but flagged as not cross-dataset comparable

In [8]:
import numpy as np
tsoi_markers = ['MITF', 'SOX10', 'NGFR', 'AXL']

stats = []
for gene in tsoi_markers:
    if gene not in adata_layer2.var.index:
        print(f'{gene}: NOT IN DATASET')
        stats.append({'gene': gene, 'in_dataset': False})
        continue

    # Get expression vector
    gene_idx = adata_layer2.var.index.get_loc(gene)
    expr = adata_layer2.X[:, gene_idx]
    if hasattr(expr, 'toarray'):
        expr = expr.toarray().flatten()
    else:
        expr = np.array(expr).flatten()

    nz = expr[expr > 0]
    stats.append({
        'gene': gene,
        'in_dataset': True,
        'mean_all': expr.mean(),
        'mean_nonzero': nz.mean() if len(nz) > 0 else 0,
        'pct_expressing': (expr > 0).mean() * 100,
        'p50': np.percentile(expr, 50),
        'p75': np.percentile(expr, 75),
        'p95': np.percentile(expr, 95),
        'max': expr.max(),
    })

stats_df = pd.DataFrame(stats)
print('Tsoi marker distribution in Layer 2 (Jerby-Arnon-New malignant, 860 cells):')
print(stats_df.to_string(index=False))

Tsoi marker distribution in Layer 2 (Jerby-Arnon-New malignant, 860 cells):
 gene  in_dataset  mean_all  mean_nonzero  pct_expressing      p50      p75      p95      max
 MITF        True  3.853188      6.014050       64.069767 5.279170 6.651653 7.564024 8.815770
SOX10        True  2.099003      4.227499       49.651163 0.000000 4.484285 5.606559 6.826786
 NGFR        True  0.373801      3.738011       10.000000 0.000000 0.000000 3.997928 6.463826
  AXL        True  2.636700      2.782285       94.767442 2.793805 3.444412 4.965284 8.118928


In [9]:
import matplotlib.pyplot as plt
Path('../results/figures').mkdir(parents=True, exist_ok=True)

# Use scanpy violin
sc.pl.violin(
    adata_layer2,
    tsoi_markers,
    jitter=0.4,
    show=False,
)
plt.savefig('../results/figures/08_layer2_tsoi_markers_violin.png',
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved violin plot: results/figures/08_layer2_tsoi_markers_violin.png')

Saved violin plot: results/figures/08_layer2_tsoi_markers_violin.png


In [10]:
fig, ax = plt.subplots(figsize=(8, 5))
gene_idx = adata_layer2.var.index.get_loc('NGFR')
expr = adata_layer2.X[:, gene_idx]
if hasattr(expr, 'toarray'):
    expr = expr.toarray().flatten()
else:
    expr = np.array(expr).flatten()

ax.hist(expr, bins=50, edgecolor='black')
ax.set_xlabel('NGFR expression (log1p normalized)')
ax.set_ylabel('Number of cells')
ax.set_title(f'NGFR distribution — Layer 2 (Jerby-Arnon-New malignant, n={len(expr)})')
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='zero')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/08_layer2_ngfr_histogram.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved NGFR histogram: results/figures/08_layer2_ngfr_histogram.png')

print('\n=== Phase 1.2 done ===')

Saved NGFR histogram: results/figures/08_layer2_ngfr_histogram.png

=== Phase 1.2 done ===


## Phase 2 — Integration + Clustering + Tsoi state annotation

Mirror Stage 2 paradigm on Layer 2 (Jerby-Arnon-New malignant, 860 cells, 11 samples).

Convention locks (per CLAUDE.md Stage 2 Working Spec):
- HVG: `flavor='seurat'`, 2000 genes
- Scale: `sc.pp.scale` (default)
- PCA: 50 components
- Harmony: `batch_key='samples'`, `theta=2` (default)
- Neighbors: `n_neighbors=15`, `n_pcs=30`
- Leiden: multi-resolution sweep {0.1, 0.2, 0.3, 0.5, 0.8, 1.0}
- Tsoi marker thresholds: re-calibrate per dataset (Layer 2 norm scheme differs from Tirosh `log2(TPM/10+1)`)

In [11]:
# QC stats already in obs (no.of.genes, no.of.reads)
print('QC stats from Jerby-Arnon annotation:')
print(adata_layer2.obs[['no.of.genes', 'no.of.reads']].describe())

# HVG selection on log-normalized data (Cell 11 already normalized)
sc.pp.highly_variable_genes(adata_layer2, flavor='seurat', n_top_genes=2000)
print(f'HVGs selected: {adata_layer2.var.highly_variable.sum()}')

# Subset to HVGs for downstream
adata_hvg = adata_layer2[:, adata_layer2.var.highly_variable].copy()
print(f'HVG-subset shape: {adata_hvg.shape}')

QC stats from Jerby-Arnon annotation:
        no.of.genes   no.of.reads
count    860.000000  8.600000e+02
mean    5411.786047  1.139525e+06
std     2428.072216  7.443978e+05
min     1713.000000  9.685000e+03
25%     3182.000000  6.690102e+05
50%     5357.000000  9.860125e+05
75%     7326.500000  1.446008e+06
max    13691.000000  7.107557e+06
extracting highly variable genes


    finished (0:00:00)


HVGs selected: 2000
HVG-subset shape: (860, 2000)


In [12]:
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=50, random_state=0)
print(f'PCA done. variance_ratio top 10: {adata_hvg.uns["pca"]["variance_ratio"][:10]}')

computing PCA


    with n_comps=50


    finished (0:00:00)


PCA done. variance_ratio top 10: [0.05223039 0.03460203 0.02720699 0.02292976 0.01400028 0.01185831
 0.01126902 0.01008092 0.00931219 0.00751571]


In [13]:
import harmonypy as hm
import logging
print(f'Harmony input: X_pca shape {adata_hvg.obsm["X_pca"].shape}')
print(f'Batch key: samples, n_batches: {adata_hvg.obs["samples"].nunique()}')

# Per Stage 2 notebook 03 working pattern (with harmonypy 0.2.0 PyTorch rewrite):
# - Z_corr is already (n_cells, n_pcs); do NOT transpose.
# - Wrap in np.ascontiguousarray to avoid negative-stride issues.
# - Pass X_pca through ascontiguousarray for input safety.
X_pca_in = np.ascontiguousarray(adata_hvg.obsm['X_pca'])

logging.disable(logging.INFO)
try:
    hm_out = hm.run_harmony(
        X_pca_in,
        adata_hvg.obs,
        'samples',
        theta=2,
        random_state=0,
        verbose=False,
    )
finally:
    logging.disable(logging.NOTSET)

adata_hvg.obsm['X_pca_harmony'] = np.ascontiguousarray(hm_out.Z_corr)
assert adata_hvg.obsm['X_pca_harmony'].shape == adata_hvg.obsm['X_pca'].shape, \
    'Harmony output shape mismatch — check harmonypy Z_corr layout'
print(f'Harmony done. X_pca_harmony shape: {adata_hvg.obsm["X_pca_harmony"].shape}')

Harmony input: X_pca shape (860, 50)
Batch key: samples, n_batches: 11


Harmony done. X_pca_harmony shape: (860, 50)


In [14]:
sc.pp.neighbors(adata_hvg, n_neighbors=15, n_pcs=30, use_rep='X_pca_harmony', random_state=0)
sc.tl.umap(adata_hvg, random_state=0)
print('UMAP done')

computing neighbors


/Users/youyouwu/miniforge3/envs/melanoma-scrnaseq/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    finished (0:00:03)


computing UMAP


    finished (0:00:01)


UMAP done


In [15]:
resolutions = [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
for res in resolutions:
    key = f'leiden_r{res}'
    sc.tl.leiden(adata_hvg, resolution=res, key_added=key, random_state=0)
    n_clusters = adata_hvg.obs[key].nunique()
    print(f'{key}: {n_clusters} clusters')

running Leiden clustering


    finished (0:00:00)


leiden_r0.1: 6 clusters
running Leiden clustering


    finished (0:00:00)


leiden_r0.2: 8 clusters
running Leiden clustering


...

    finished (0:00:00)


leiden_r0.8: 14 clusters
running Leiden clustering


    finished (0:00:00)


leiden_r1.0: 15 clusters


/var/folders/v7/sk64vbls6fg9sxpscqd5t8jw0000gn/T/ipykernel_24893/3619997586.py:4: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata_hvg, resolution=res, key_added=key, random_state=0)


In [16]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Patient
sc.pl.umap(adata_hvg, color='samples', ax=axes[0], show=False, legend_loc='on data', legend_fontsize=6)
axes[0].set_title('Layer 2 UMAP by patient')

# leiden_r0.3 (Stage 2 working resolution, exploratory)
sc.pl.umap(adata_hvg, color='leiden_r0.3', ax=axes[1], show=False, legend_loc='on data')
axes[1].set_title('Leiden r=0.3')

# leiden_r0.5 (BBKNN-style working resolution from Q3.2, exploratory)
sc.pl.umap(adata_hvg, color='leiden_r0.5', ax=axes[2], show=False, legend_loc='on data')
axes[2].set_title('Leiden r=0.5')

plt.tight_layout()
plt.savefig('../results/figures/08_layer2_umap_overview.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved UMAP overview: results/figures/08_layer2_umap_overview.png')

Saved UMAP overview: results/figures/08_layer2_umap_overview.png


In [17]:
tsoi_markers = ['MITF', 'SOX10', 'NGFR', 'AXL']

# HVG selection kept AXL but dropped MITF/SOX10/NGFR (low variance — NGFR is
# only 10% expressing). Plot dotplots on adata_layer2 (full var) using Leiden
# labels computed on adata_hvg. Cell order is preserved (adata_hvg was created
# by boolean var-mask, not row reordering).
assert (adata_hvg.obs_names == adata_layer2.obs_names).all(), 'cell order mismatch'
for res in [0.1, 0.2, 0.3, 0.5, 0.8, 1.0]:
    key = f'leiden_r{res}'
    adata_layer2.obs[key] = adata_hvg.obs[key].values

# Per-resolution dotplot on adata_layer2 (so all 4 markers are accessible)
for res in [0.1, 0.2, 0.3, 0.5, 0.8]:
    key = f'leiden_r{res}'
    sc.pl.dotplot(adata_layer2, tsoi_markers, groupby=key,
                  standard_scale='var', show=False)
    plt.savefig(f'../results/figures/08_layer2_dotplot_{key}.png',
                dpi=150, bbox_inches='tight')
    plt.close()
print('Saved per-resolution dotplots')

Saved per-resolution dotplots


In [18]:
# Compute marker stats per cluster for all sweep resolutions
for res in [0.1, 0.2, 0.3, 0.5, 0.8]:
    key = f'leiden_r{res}'
    print(f'\n=== {key} ===')
    for gene in tsoi_markers:
        gene_idx = adata_hvg.var.index.get_loc(gene) if gene in adata_hvg.var.index else None
        if gene_idx is None:
            # HVG subset may not include the gene; fall back to full adata_layer2
            if gene in adata_layer2.var.index:
                gene_idx_full = adata_layer2.var.index.get_loc(gene)
                expr = adata_layer2.X[:, gene_idx_full]
                if hasattr(expr, 'toarray'):
                    expr = expr.toarray().flatten()
                else:
                    expr = np.array(expr).flatten()
                per_cluster = pd.DataFrame({
                    'cluster': adata_hvg.obs[key].values,
                    'expr': expr,
                })
            else:
                print(f'  {gene}: NOT IN DATASET')
                continue
        else:
            expr = adata_hvg.X[:, gene_idx]
            if hasattr(expr, 'toarray'):
                expr = expr.toarray().flatten()
            else:
                expr = np.array(expr).flatten()
            per_cluster = pd.DataFrame({
                'cluster': adata_hvg.obs[key].values,
                'expr': expr,
            })

        agg = per_cluster.groupby('cluster')['expr'].agg([
            ('mean', 'mean'),
            ('pct_expr', lambda x: (x > 0).mean() * 100),
        ])
        print(f'  {gene}:')
        print(agg.to_string())


=== leiden_r0.1 ===
  MITF:
             mean   pct_expr
cluster                     
0        3.025317  49.122807
1        5.023464  84.501845
2        6.296260  94.285714
3        0.988674  26.388889
4        3.210401  50.000000
5        1.806989  53.571429
  SOX10:
             mean   pct_expr
cluster                     
0        1.442119  34.502924
1        3.229123  76.752768
2        3.674090  84.761905
3        0.134413   2.777778
4        0.741721  16.666667
5        0.365568  10.714286
  NGFR:
             mean   pct_expr
cluster                     
0        0.108138   2.339181
1        0.167430   5.166052
2        0.057198   0.952381
3        2.183740  61.111111
4        0.190506   4.761905
5        2.424144  60.714286
  AXL:
             mean    pct_expr
cluster                      
0       -0.644282   18.713450
1        0.430343   80.442804
2        0.668531   99.047619
3       -0.034148   36.111111
4        1.001772  100.000000
5       -0.217502   50.000000

=== leiden

/var/folders/v7/sk64vbls6fg9sxpscqd5t8jw0000gn/T/ipykernel_24893/65687701.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg = per_cluster.groupby('cluster')['expr'].agg([
/var/folders/v7/sk64vbls6fg9sxpscqd5t8jw0000gn/T/ipykernel_24893/65687701.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg = per_cluster.groupby('cluster')['expr'].agg([
/var/folders/v7/sk64vbls6fg9sxpscqd5t8jw0000gn/T/ipykernel_24893/65687701.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

In [19]:
# Key question: where are the 86 NGFR > 0 cells distributed across clusters?
gene_idx_full = adata_layer2.var.index.get_loc('NGFR')
ngfr_expr = adata_layer2.X[:, gene_idx_full]
if hasattr(ngfr_expr, 'toarray'):
    ngfr_expr = ngfr_expr.toarray().flatten()
else:
    ngfr_expr = np.array(ngfr_expr).flatten()

ngfr_positive_mask = ngfr_expr > 0
print(f'NGFR-positive cells: {ngfr_positive_mask.sum()} (10.0%)')

# Distribution across clusters per sweep resolution
for res in [0.1, 0.2, 0.3, 0.5, 0.8]:
    key = f'leiden_r{res}'
    cluster_assign = adata_hvg.obs[key].values
    ngfr_cluster_dist = pd.Series(cluster_assign[ngfr_positive_mask]).value_counts(normalize=False)
    total_cluster_dist = pd.Series(cluster_assign).value_counts()

    print(f'\n=== {key} — NGFR-positive distribution across clusters ===')
    for cluster in total_cluster_dist.index:
        n_ngfr = ngfr_cluster_dist.get(cluster, 0)
        n_total = total_cluster_dist[cluster]
        pct_ngfr_in_cluster = (n_ngfr / n_total) * 100
        print(f'  Cluster {cluster}: {n_ngfr}/{n_total} = {pct_ngfr_in_cluster:.1f}% NGFR+')

NGFR-positive cells: 86 (10.0%)

=== leiden_r0.1 — NGFR-positive distribution across clusters ===
  Cluster 0: 8/342 = 2.3% NGFR+
  Cluster 1: 14/271 = 5.2% NGFR+
  Cluster 2: 1/105 = 1.0% NGFR+
  Cluster 3: 44/72 = 61.1% NGFR+
  Cluster 4: 2/42 = 4.8% NGFR+
  Cluster 5: 17/28 = 60.7% NGFR+

=== leiden_r0.2 — NGFR-positive distribution across clusters ===
  Cluster 0: 8/336 = 2.4% NGFR+
  Cluster 1: 5/142 = 3.5% NGFR+
  Cluster 2: 4/111 = 3.6% NGFR+
  Cluster 3: 1/105 = 1.0% NGFR+
  Cluster 4: 44/72 = 61.1% NGFR+
  Cluster 5: 2/42 = 4.8% NGFR+
  Cluster 6: 17/28 = 60.7% NGFR+
  Cluster 7: 5/24 = 20.8% NGFR+

=== leiden_r0.3 — NGFR-positive distribution across clusters ===
  Cluster 0: 8/335 = 2.4% NGFR+
  Cluster 1: 5/142 = 3.5% NGFR+
  Cluster 2: 4/111 = 3.6% NGFR+
  Cluster 3: 1/105 = 1.0% NGFR+
  Cluster 4: 44/72 = 61.1% NGFR+
  Cluster 5: 2/42 = 4.8% NGFR+
  Cluster 6: 17/29 = 58.6% NGFR+
  Cluster 7: 5/24 = 20.8% NGFR+

=== leiden_r0.5 — NGFR-positive distribution across clusters 

## Phase 2.5 — Tsoi state annotation (working resolution r=0.1)

Per Phase 2 data review:
- Working resolution: `leiden_r0.1` (6 clusters, parsimonious 4-state + 2 ambiguous)
- Tsoi state assignment based on dotplot + per-cluster `pct_expressing`
- NOT directly transferring Tirosh thresholds (different normalization scheme)

### Cluster → Tsoi state mapping (manually assigned per dotplot + stats)

| Cluster | n | MITF% | SOX10% | NGFR% | AXL pattern | Tsoi state |
|---|---|---|---|---|---|---|
| 0 | 342 | 49% | 35% | 2.3% | broad/wide | Ambiguous-Mel |
| 1 | 271 | 85% | 77% | 5.2% | moderate | Melanocytic |
| 2 | 105 | 94% | 85% | 1.0% | moderate | Melanocytic (strong) |
| 3 | 72 | 26% | 2.8% | 61.1% | scaled-high | Neural crest-like |
| 4 | 42 | 50% | 17% | 4.8% | moderate | Ambiguous |
| 5 | 28 | 54% | 11% | 60.7% | scaled-mod | Transitory |

**Annotation rationale:**
- Cluster 1 + 2 collapse to "Melanocytic" (both MITF/SOX10 high, distinction is degree not category — preserved as separate clusters for Stage 2 paradigm consistency)
- Cluster 0: high MITF% (49%) but mixed — labeled Ambiguous-Mel to flag uncertainty between Mel and dedifferentiated continuum
- Cluster 3: Neural crest-like by Tsoi definition (MITF↓, SOX10↓ rarely, NGFR↑, AXL↑) — matches Tsoi NC marker profile despite low MITF
- Cluster 4: 50% MITF without SOX10 backup, weak NGFR — labeled Ambiguous
- Cluster 5: Transitory by Tsoi definition (MITF↑ + NGFR↑) — matches Tsoi Transitory marker profile

In [20]:
# Cluster → Tsoi state mapping
tsoi_state_map = {
    '0': 'Ambiguous-Mel',
    '1': 'Melanocytic',
    '2': 'Melanocytic',
    '3': 'Neural crest-like',
    '4': 'Ambiguous',
    '5': 'Transitory',
}

# Tsoi ordering for downstream display (M -> T -> NC -> ambiguous bins)
tsoi_order = ['Melanocytic', 'Transitory', 'Neural crest-like',
              'Ambiguous-Mel', 'Ambiguous']

# Apply on adata_hvg
state_values = (
    adata_hvg.obs['leiden_r0.1']
    .astype(str)
    .map(tsoi_state_map)
)
adata_hvg.obs['tsoi_state'] = pd.Categorical(state_values, categories=tsoi_order)

# Mirror to adata_layer2
adata_layer2.obs['tsoi_state'] = pd.Categorical(
    adata_hvg.obs['tsoi_state'].astype(str).values,
    categories=tsoi_order,
)

print('Tsoi state distribution in Layer 2:')
print(adata_hvg.obs['tsoi_state'].value_counts())

Tsoi state distribution in Layer 2:
tsoi_state
Melanocytic          376
Ambiguous-Mel        342
Neural crest-like     72
Ambiguous             42
Transitory            28
Name: count, dtype: int64


In [21]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: leiden_r0.1
sc.pl.umap(adata_hvg, color='leiden_r0.1', ax=axes[0], show=False,
           legend_loc='on data', legend_fontsize=10)
axes[0].set_title('Leiden r=0.1 (working resolution)')

# Right: Tsoi state
sc.pl.umap(adata_hvg, color='tsoi_state', ax=axes[1], show=False,
           legend_loc='right margin', legend_fontsize=10)
axes[1].set_title('Layer 2 Tsoi state annotation')

plt.tight_layout()
plt.savefig('../results/figures/08_layer2_tsoi_state_annotated.png',
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved Tsoi state UMAP: results/figures/08_layer2_tsoi_state_annotated.png')

Saved Tsoi state UMAP: results/figures/08_layer2_tsoi_state_annotated.png


In [22]:
tsoi_markers = ['MITF', 'SOX10', 'NGFR', 'AXL']

sc.pl.dotplot(adata_layer2, tsoi_markers, groupby='tsoi_state',
              standard_scale='var', show=False)
plt.savefig('../results/figures/08_layer2_tsoi_state_dotplot.png',
            dpi=150, bbox_inches='tight')
plt.close()
print('Saved Tsoi state dotplot: results/figures/08_layer2_tsoi_state_dotplot.png')

Saved Tsoi state dotplot: results/figures/08_layer2_tsoi_state_dotplot.png


## Phase 2.6 — Treatment enrichment hypothesis check

Treatment composition in Layer 2 (overall):
- post.treatment: 563 cells (65.5%)
- treatment.naive: 297 cells (34.5%)

Hypothesis: NGFR-high clusters (cluster 3 = NC-like, cluster 5 = Transitory)
are enriched for post-treatment cells. If supported, this would suggest
Stage 4's NGFR-defined state recovery is partly driven by Layer 2's
treatment-exposed cohort enrichment, not pure dataset/method difference
from Tirosh.

If null (NGFR-high clusters have similar treatment composition to overall):
the "Tirosh-specific failure to recover NGFR states" framing is more
purely about dataset/normalization, not treatment.

In [23]:
# Treatment x Tsoi state contingency
ct = pd.crosstab(
    adata_layer2.obs['tsoi_state'],
    adata_layer2.obs['treatment.group'],
    margins=True,
    margins_name='Total',
)
print('Tsoi state x treatment.group contingency:')
print(ct)
print()

# Pct post-treatment per Tsoi state
print('Pct post-treatment per Tsoi state:')
for state in adata_layer2.obs['tsoi_state'].cat.categories:
    mask = adata_layer2.obs['tsoi_state'] == state
    n_total = int(mask.sum())
    if n_total == 0:
        print(f'  {state}: 0 cells (skipped)')
        continue
    n_post = int((mask & (adata_layer2.obs['treatment.group'] == 'post.treatment')).sum())
    pct = (n_post / n_total) * 100
    print(f'  {state}: {n_post}/{n_total} = {pct:.1f}% post-treatment')

print(f'\nOverall baseline: 65.5% post-treatment')

Tsoi state x treatment.group contingency:
treatment.group    post.treatment  treatment.naive  Total
tsoi_state                                               
Melanocytic                   254              122    376
Transitory                     21                7     28
Neural crest-like              72                0     72
Ambiguous-Mel                 184              158    342
Ambiguous                      32               10     42
Total                         563              297    860

Pct post-treatment per Tsoi state:
  Melanocytic: 254/376 = 67.6% post-treatment
  Transitory: 21/28 = 75.0% post-treatment
  Neural crest-like: 72/72 = 100.0% post-treatment
  Ambiguous-Mel: 184/342 = 53.8% post-treatment
  Ambiguous: 32/42 = 76.2% post-treatment

Overall baseline: 65.5% post-treatment


In [24]:
from scipy.stats import chi2_contingency

# 2x2 contingency: NGFR-high (NC + Transitory) vs rest, post-treatment vs naive
ngfr_high_mask = adata_layer2.obs['tsoi_state'].isin(['Neural crest-like', 'Transitory'])
treatment_post_mask = adata_layer2.obs['treatment.group'] == 'post.treatment'

ct_2x2 = pd.crosstab(
    ngfr_high_mask.rename('NGFR_high_cluster'),
    treatment_post_mask.rename('post_treatment'),
)
print('NGFR-high (NC + Transitory) vs rest, treatment exposure:')
print(ct_2x2)

chi2, pval, dof, expected = chi2_contingency(ct_2x2)
print(f'\nChi-square test: chi2={chi2:.3f}, p={pval:.4f}, dof={dof}')
print(f'Expected counts under null (no enrichment):')
print(pd.DataFrame(expected, index=ct_2x2.index, columns=ct_2x2.columns).round(1))

print()
print('Interpretation:')
print('  p < 0.05 -> reject null, NGFR-high clusters significantly differ in treatment composition')
print('  p >= 0.05 -> cannot reject null, treatment composition similar to overall')

NGFR-high (NC + Transitory) vs rest, treatment exposure:
post_treatment     False  True 
NGFR_high_cluster              
False                290    470
True                   7     93

Chi-square test: chi2=36.582, p=0.0000, dof=1
Expected counts under null (no enrichment):
post_treatment     False  True 
NGFR_high_cluster              
False              262.5  497.5
True                34.5   65.5

Interpretation:
  p < 0.05 -> reject null, NGFR-high clusters significantly differ in treatment composition
  p >= 0.05 -> cannot reject null, treatment composition similar to overall


In [25]:
# Save annotated Layer 2 (HVG-subset, with leiden + tsoi_state + harmony embedding)
adata_hvg.write('../data/processed/jerby_arnon_layer2_annotated.h5ad')
print('Saved annotated Layer 2 HVG h5ad')

# Also save full Layer 2 (all genes preserved) with annotations
adata_layer2.write('../data/processed/jerby_arnon_layer2_full_annotated.h5ad')
print('Saved annotated Layer 2 full h5ad')

print('\n=== Phase 2.5 + 2.6 done ===')

Saved annotated Layer 2 HVG h5ad


Saved annotated Layer 2 full h5ad

=== Phase 2.5 + 2.6 done ===


## Phase 2.7 — Patient confounding + Tirosh treatment status

Phase 2.6 surfaced "NC + Transitory 100% post-treatment, χ² p < 0.0001".
Before committing this as Stage 4 finding, verify:

1. Patient breakdown: NC + Trans cells dominate by 1-2 patient? Or distributed?
2. Treatment per patient: per-patient unique or mixed (paired)?
3. Tirosh treatment status: can we verify Tirosh cohort is treatment-naive?

In [26]:
print('=== Patient distribution per Tsoi state ===')
for state in adata_layer2.obs['tsoi_state'].cat.categories:
    mask = adata_layer2.obs['tsoi_state'] == state
    print(f'\n{state} (n={mask.sum()}):')
    sample_dist = adata_layer2.obs.loc[mask, 'samples'].value_counts()
    print(sample_dist)
    print(f'  Number of distinct samples: {len(sample_dist)}')
    print(f'  Primary sample fraction: {sample_dist.iloc[0] / mask.sum() * 100:.1f}%')

=== Patient distribution per Tsoi state ===

Melanocytic (n=376):
samples
Mel102      144
Mel103       89
Mel194       48
Mel98        36
Mel110       25
Mel129pa     21
Mel105        7
Mel112        4
Mel106        1
Mel128        1
Mel121.1      0
Name: count, dtype: int64
  Number of distinct samples: 11
  Primary sample fraction: 38.3%

Transitory (n=28):
samples
Mel110      20
Mel103       6
Mel98        1
Mel105       1
Mel102       0
Mel106       0
Mel112       0
Mel121.1     0
Mel128       0
Mel129pa     0
Mel194       0
Name: count, dtype: int64
  Number of distinct samples: 11
  Primary sample fraction: 71.4%

Neural crest-like (n=72):
samples
Mel110      54
Mel106      16
Mel98        1
Mel102       1
Mel103       0
Mel105       0
Mel112       0
Mel121.1     0
Mel128       0
Mel129pa     0
Mel194       0
Name: count, dtype: int64
  Number of distinct samples: 11
  Primary sample fraction: 75.0%

Ambiguous-Mel (n=342):
samples
Mel106      75
Mel129pa    65
Mel194      48
Mel1

In [27]:
print('=== Treatment composition per sample ===')

# Per sample, treatment composition
sample_treatment = pd.crosstab(
    adata_layer2.obs['samples'],
    adata_layer2.obs['treatment.group'],
)
print(sample_treatment)
print()

# Sample = post-only / naive-only / mixed?
for sample in sample_treatment.index:
    row = sample_treatment.loc[sample]
    naive_n = row.get('treatment.naive', 0)
    post_n = row.get('post.treatment', 0)
    total = naive_n + post_n
    if naive_n == 0:
        status = '100% POST'
    elif post_n == 0:
        status = '100% NAIVE'
    else:
        status = f'MIXED ({naive_n} naive, {post_n} post)'
    print(f'{sample} (n={total}): {status}')

=== Treatment composition per sample ===
treatment.group  post.treatment  treatment.naive
samples                                         
Mel98                        76                0
Mel102                      169                0
Mel103                        0              137
Mel105                        0               14
Mel106                       94                0
Mel110                      123                0
Mel112                        0               28
Mel121.1                      5                0
Mel128                        0               26
Mel129pa                      0               92
Mel194                       96                0

Mel98 (n=76): 100% POST
Mel102 (n=169): 100% POST
Mel103 (n=137): 100% NAIVE
Mel105 (n=14): 100% NAIVE
Mel106 (n=94): 100% POST
Mel110 (n=123): 100% POST
Mel112 (n=28): 100% NAIVE
Mel121.1 (n=5): 100% POST
Mel128 (n=26): 100% NAIVE
Mel129pa (n=92): 100% NAIVE
Mel194 (n=96): 100% POST


In [28]:
# adata_full contains Tirosh cohort cells; check their treatment.group
print('=== Treatment.group in Tirosh cohort (per Jerby-Arnon annotation) ===')
tirosh_mask = adata_full.obs['Cohort'] == 'Tirosh'
print(f'Tirosh cohort size: {tirosh_mask.sum()}')
print()
print('treatment.group distribution in Tirosh cohort:')
print(adata_full.obs.loc[tirosh_mask, 'treatment.group'].value_counts(dropna=False))
print()

# Comparison with New cohort
new_mask = adata_full.obs['Cohort'] == 'New'
print('treatment.group distribution in New cohort:')
print(adata_full.obs.loc[new_mask, 'treatment.group'].value_counts(dropna=False))

=== Treatment.group in Tirosh cohort (per Jerby-Arnon annotation) ===
Tirosh cohort size: 4199

treatment.group distribution in Tirosh cohort:
treatment.group
treatment.naive    2396
post.treatment     1803
Name: count, dtype: int64

treatment.group distribution in New cohort:
treatment.group
post.treatment     1753
treatment.naive    1234
Name: count, dtype: int64


In [29]:
print('=== SYNTHESIS ===')
print()
print('Per Phase 2.6 finding: NC 100% post, Trans 75% post')
print()
print('Phase 2.7 checks tell us:')
print('1. Are NC cells from 1-2 patients (confound) or distributed (robust)?')
print('2. Is per-patient treatment uniform (full confound) or mixed (separable)?')
print('3. Does Jerby-Arnon label Tirosh cohort cells as naive/post/unknown?')

# NC + Transitory cell counts per patient
ngfr_high_mask = adata_layer2.obs['tsoi_state'].isin(['Neural crest-like', 'Transitory'])
print(f'\nNGFR-high (NC + Trans) cells per sample:')
ngfr_high_per_sample = adata_layer2.obs.loc[ngfr_high_mask, 'samples'].value_counts()
print(ngfr_high_per_sample)
print(f'\nDistinct samples contributing to NGFR-high: {len(ngfr_high_per_sample)}')
print(f'Top sample fraction: {ngfr_high_per_sample.iloc[0] / ngfr_high_mask.sum() * 100:.1f}%')

=== SYNTHESIS ===

Per Phase 2.6 finding: NC 100% post, Trans 75% post

Phase 2.7 checks tell us:
1. Are NC cells from 1-2 patients (confound) or distributed (robust)?
2. Is per-patient treatment uniform (full confound) or mixed (separable)?
3. Does Jerby-Arnon label Tirosh cohort cells as naive/post/unknown?

NGFR-high (NC + Trans) cells per sample:
samples
Mel110      74
Mel106      16
Mel103       6
Mel98        2
Mel102       1
Mel105       1
Mel112       0
Mel121.1     0
Mel128       0
Mel129pa     0
Mel194       0
Name: count, dtype: int64

Distinct samples contributing to NGFR-high: 11
Top sample fraction: 74.0%


## Phase 2.8 — Cross-validating Stage 3 P1 against patient heterogeneity

Phase 2.7 surfaced that Layer 2 NGFR-high subpopulations are 74% from single patient Mel110. This raises a question about Stage 3 P1 itself:

Stage 3 P1 was based on Stage 2 Tirosh malignant cells (n=1257, Tirosh 2016 GSE72056). The claim was "dataset-level NGFR weakness". But Stage 3 did not analyze patient-level breakdown.

Question: Do the Stage 2 1257 Tirosh malignant cells, when looked up in Jerby-Arnon's re-annotation of the same Tirosh dataset, show:
- Patient distribution: dominated by 1-2 patients, or diverse?
- Treatment composition: naive-dominant, post-dominant, or mixed?
- Any NGFR-positive cells, and from which patients?

This verifies whether Stage 3 P1's "dataset-level" framing was also potentially subject to the same epistemic concern Phase 2.7 raised about Layer 2.

In [30]:
# Approach: Stage 2 used Tirosh malignant cells (malignant=2 per Tirosh's metadata).
# Jerby-Arnon re-annotated cell.types. We find:
# - Jerby-Arnon Cohort='Tirosh' cells
# - Where cell.types == 'Mal' (Jerby-Arnon's malignant judgement)
# This gives the closest mapping to Stage 2's 1257 (not identical filter:
# Tirosh malignant_status vs Jerby-Arnon Mal — operationally the cleanest
# cross-reference).

tirosh_mal_in_ja = adata_full[
    (adata_full.obs['Cohort'] == 'Tirosh') &
    (adata_full.obs['cell.types'] == 'Mal')
].copy()

print(f'Jerby-Arnon Tirosh cohort x Mal cells: {tirosh_mal_in_ja.n_obs}')
print(f'(Stage 2 reported 1257 Tirosh malignant cells)')
print()

# RAW-COUNTS SANITY CHECK before any normalization in Cell 43
# adata_full was loaded from GSE115978_counts.csv.gz and never normalized,
# but verify it is in fact raw counts (X.max() should be >> 20).
print('--- raw-counts sanity check on adata_full / tirosh_mal_in_ja ---')
print(f'  X dtype: {tirosh_mal_in_ja.X.dtype}')
print(f'  X.min(): {tirosh_mal_in_ja.X.min()}')
print(f'  X.max(): {tirosh_mal_in_ja.X.max()}')
print(f'  X.mean(): {tirosh_mal_in_ja.X.mean():.3f}')
# Expectation: max in the hundreds-to-thousands range for Smart-seq2 read counts.
# If max < 20 -> already normalized somewhere -> surface and DO NOT re-normalize.
if tirosh_mal_in_ja.X.max() < 20:
    raise RuntimeError(
        f'X.max() = {tirosh_mal_in_ja.X.max()} < 20 -> tirosh_mal_in_ja '
        f'appears already normalized. Surface to Ewan before continuing.'
    )
print('  PASSED: looks like raw counts, safe to normalize in Cell 43.')

Jerby-Arnon Tirosh cohort x Mal cells: 1158
(Stage 2 reported 1257 Tirosh malignant cells)

--- raw-counts sanity check on adata_full / tirosh_mal_in_ja ---
  X dtype: float32
  X.min(): 0.0
  X.max(): 159721.0
  X.mean(): 8.812
  PASSED: looks like raw counts, safe to normalize in Cell 43.


In [31]:
print('=== Patient (samples) distribution in Tirosh x Mal cells ===')
patient_dist = tirosh_mal_in_ja.obs['samples'].value_counts()
print(patient_dist)
print()
print(f'Distinct patients: {len(patient_dist)}')
print(f'Top patient fraction: {patient_dist.iloc[0] / tirosh_mal_in_ja.n_obs * 100:.1f}%')
print(f'Top 3 patients fraction: {patient_dist.iloc[:3].sum() / tirosh_mal_in_ja.n_obs * 100:.1f}%')

=== Patient (samples) distribution in Tirosh x Mal cells ===
samples
Mel79    486
Mel88    124
Mel78    120
Mel81    120
Mel89    100
Mel80     92
Mel71     62
Mel84     16
Mel53     14
Mel60     12
Mel94      6
Mel82      6
Name: count, dtype: int64

Distinct patients: 12
Top patient fraction: 42.0%
Top 3 patients fraction: 63.0%


In [32]:
print('=== Treatment.group composition in Tirosh x Mal cells ===')
treatment_dist = tirosh_mal_in_ja.obs['treatment.group'].value_counts(dropna=False)
print(treatment_dist)
print()

# Per-patient treatment uniformity check (same lens as Layer 2 Phase 2.7)
print('Per-patient treatment composition (Tirosh cohort):')
per_pt = pd.crosstab(
    tirosh_mal_in_ja.obs['samples'],
    tirosh_mal_in_ja.obs['treatment.group'],
)
for sample in per_pt.index:
    row = per_pt.loc[sample]
    naive_n = row.get('treatment.naive', 0)
    post_n = row.get('post.treatment', 0)
    total = naive_n + post_n
    if naive_n == 0:
        status = '100% POST'
    elif post_n == 0:
        status = '100% NAIVE'
    else:
        status = f'MIXED ({naive_n} naive, {post_n} post)'
    print(f'{sample} (n={total}): {status}')

=== Treatment.group composition in Tirosh x Mal cells ===
treatment.group
treatment.naive    896
post.treatment     262
Name: count, dtype: int64

Per-patient treatment composition (Tirosh cohort):
Mel53 (n=14): 100% NAIVE
Mel60 (n=12): 100% POST
Mel71 (n=62): 100% NAIVE
Mel78 (n=120): 100% POST
Mel79 (n=486): 100% NAIVE
Mel80 (n=92): 100% NAIVE
Mel81 (n=120): 100% NAIVE
Mel82 (n=6): 100% NAIVE
Mel84 (n=16): 100% NAIVE
Mel88 (n=124): 100% POST
Mel89 (n=100): 100% NAIVE
Mel94 (n=6): 100% POST


In [33]:
# Normalize Tirosh cohort x Mal (same scheme as Layer 2: CP10K + log1p — wait,
# Layer 2 used target_sum=1e6 (CPM) + log1p; mirror that for comparability)
tirosh_mal_norm = tirosh_mal_in_ja.copy()
sc.pp.normalize_total(tirosh_mal_norm, target_sum=1e6)
sc.pp.log1p(tirosh_mal_norm)

# Find NGFR expression
gene_idx = tirosh_mal_norm.var.index.get_loc('NGFR')
ngfr_expr = tirosh_mal_norm.X[:, gene_idx]
if hasattr(ngfr_expr, 'toarray'):
    ngfr_expr = ngfr_expr.toarray().flatten()
else:
    ngfr_expr = np.array(ngfr_expr).flatten()

ngfr_positive_mask = ngfr_expr > 0
n_pos = int(ngfr_positive_mask.sum())
print(f'NGFR-positive cells in Tirosh x Mal: {n_pos} / {tirosh_mal_norm.n_obs} = {n_pos/tirosh_mal_norm.n_obs*100:.1f}%')
print()

# Which patients?
print('NGFR-positive cells per patient (Tirosh x Mal):')
ngfr_pos_patients = tirosh_mal_norm.obs.loc[ngfr_positive_mask, 'samples'].value_counts()
print(ngfr_pos_patients)
print()
print(f'Distinct patients with NGFR+: {len(ngfr_pos_patients)}')
if len(ngfr_pos_patients) > 0:
    print(f'Top patient fraction of NGFR+: {ngfr_pos_patients.iloc[0] / n_pos * 100:.1f}%')

normalizing counts per cell


    finished (0:00:00)


NGFR-positive cells in Tirosh x Mal: 98 / 1158 = 8.5%

NGFR-positive cells per patient (Tirosh x Mal):
samples
Mel79    31
Mel89    26
Mel81    20
Mel53    14
Mel80     2
Mel82     2
Mel94     1
Mel78     1
Mel88     1
Name: count, dtype: int64

Distinct patients with NGFR+: 9
Top patient fraction of NGFR+: 31.6%


In [34]:
print('=== SYNTHESIS: patient confounding in NGFR-high cells ===\n')

print('LAYER 2 (Jerby-Arnon-New malignant, n=860):')
# Layer 2 NGFR+ patient distribution
gene_idx_l2 = adata_layer2.var.index.get_loc('NGFR')
ngfr_l2 = adata_layer2.X[:, gene_idx_l2]
if hasattr(ngfr_l2, 'toarray'):
    ngfr_l2 = ngfr_l2.toarray().flatten()
else:
    ngfr_l2 = np.array(ngfr_l2).flatten()
ngfr_l2_mask = ngfr_l2 > 0
l2_ngfr_patients = adata_layer2.obs.loc[ngfr_l2_mask, 'samples'].value_counts()
n_pos_l2 = int(ngfr_l2_mask.sum())
print(f'  NGFR+ overall: {n_pos_l2} cells ({n_pos_l2/adata_layer2.n_obs*100:.1f}%)')
print(f'  NGFR+ per patient (Layer 2):')
print(l2_ngfr_patients)
print(f'  Distinct patients with NGFR+ in Layer 2: {len(l2_ngfr_patients)}')
print(f'  Top patient fraction: {l2_ngfr_patients.iloc[0]/n_pos_l2*100:.1f}%')
print()

print('TIROSH (Jerby-Arnon\'s annotation of Tirosh malignant):')
print(f'  NGFR+ overall: {n_pos} cells ({n_pos/tirosh_mal_norm.n_obs*100:.1f}%)')
if len(ngfr_pos_patients) > 0:
    print(f'  Distinct patients with NGFR+: {len(ngfr_pos_patients)}')
    print(f'  Top patient fraction: {ngfr_pos_patients.iloc[0]/n_pos*100:.1f}%')
print()

print('INTERPRETATION GUIDE:')
print('  - If Tirosh NGFR+ also dominated by 1-2 patients: P1 itself patient-confounded')
print('  - If Tirosh NGFR+ distributed across many patients: P1 robust to patient effect')
print('  - Cross-dataset comparison limited if both have patient confounding')

=== SYNTHESIS: patient confounding in NGFR-high cells ===

LAYER 2 (Jerby-Arnon-New malignant, n=860):
  NGFR+ overall: 86 cells (10.0%)
  NGFR+ per patient (Layer 2):
samples
Mel110      78
Mel105       3
Mel194       2
Mel98        1
Mel102       1
Mel103       1
Mel106       0
Mel112       0
Mel121.1     0
Mel128       0
Mel129pa     0
Name: count, dtype: int64
  Distinct patients with NGFR+ in Layer 2: 11
  Top patient fraction: 90.7%

TIROSH (Jerby-Arnon's annotation of Tirosh malignant):
  NGFR+ overall: 98 cells (8.5%)
  Distinct patients with NGFR+: 9
  Top patient fraction: 31.6%

INTERPRETATION GUIDE:
  - If Tirosh NGFR+ also dominated by 1-2 patients: P1 itself patient-confounded
  - If Tirosh NGFR+ distributed across many patients: P1 robust to patient effect
  - Cross-dataset comparison limited if both have patient confounding


## Phase 2 — Summary

Stage 4 Layer 2 (Jerby-Arnon-New malignant, n=860 cells, 11 patients) applies Stage 2-style Harmony + Leiden pipeline at working resolution r=0.1 (6 clusters), mapping cluster → Tsoi state via marker dotplot:

| Tsoi state | n | Notes |
|---|---|---|
| Melanocytic | 376 | clusters 1+2 (MITF/SOX10 high) |
| Ambiguous-Mel | 342 | cluster 0 (mid-gradient) |
| Neural crest-like | 72 | cluster 3 (NGFR 61%, MITF/SOX10 low) |
| Ambiguous | 42 | cluster 4 (AXL high, mixed weak) |
| Transitory | 28 | cluster 5 (MITF mid, NGFR 61%) |

### Layer 2 finding (final, after Phase 2.5–2.8)

Layer 2 recovers small NGFR-high subpopulations (NC + Transitory, n=100 combined). Sanity checks (Phase 2.7–2.8) reveal these are **dominated by patient identity, not treatment**:

- NC + Trans cells: 74% from Mel110 alone (91% of all Layer 2 NGFR+)
- Mel110 is a post-treatment tumor, but treatment is fully confounded with patient identity in Layer 2 (no within-patient paired samples)
- Cross-cohort check on Stage 2's Tirosh malignant subset (mapped via Jerby-Arnon Tirosh-cohort annotation, 1158 cells): NGFR+ rate similar (8.5%), but **distributed across 9 patients** (top patient only 32%); no Mel110-equivalent NGFR-rich tumor present in Tirosh

### Stage 3 P1 — refined, not refuted

Stage 3 P1 ("Tirosh NGFR uniformly weak across malignant cells, preventing Trans/NC recovery") survives patient-level scrutiny — Tirosh NGFR+ cells spread across 9 patients, not driven by 1-2 outliers. P1's cohort-level finding is correct.

Layer 2 contribution: **mechanistic refinement of P1**. The Tirosh limitation is not uniform NGFR suppression across all cells, but rather **cohort composition** — absence of NGFR-rich tumors (like Mel110 in Layer 2, or Mel53 in Tirosh with only 14 cells, too few for stable cluster formation) that would enable NGFR-defined cluster formation. NGFR-defined Tsoi states in melanoma scRNA-seq appear tumor-specific rather than uniformly distributed; recoverability at cohort level depends on whether NGFR-rich tumors are sampled.

### Caveats explicitly documented

- Layer 2 NGFR-high cluster recovery (n=100) rests on 2 patients (Mel110 + Mel106). Stage 4 cannot statistically attribute this to treatment, dataset, or normalization — patient identity is the confound (decision log entry i).
- "Tumor-specific" hypothesis is raised by 2 tumors (Mel53, Mel110). n=2 patients is hypothesis-generating, not confirmatory.
- Treatment exposure cannot be ruled in or out as a contributing mechanism; the design simply cannot separate it from per-tumor heterogeneity.

### Methodological note

This Layer 2 analysis went through 4 verdict revisions (Phase 1.2 → 2 → 2.6 → 2.8). Each phase's data forced a refined interpretation of the previous phase's verdict. Decision log entry (i) documents the Phase 2.6 mentor naïveté (χ² narrative reach) that Phase 2.7 sanity checks corrected.